In [1]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("muhammadimran112233/employees-evaluation-for-promotion")

print("Path to dataset files:", path)
df = pd.read_csv(os.path.join(path, "employee_promotion.csv"))
df.head()

Path to dataset files: /kaggle/input/datasets/muhammadimran112233/employees-evaluation-for-promotion


,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49.0,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60.0,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50.0,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50.0,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73.0,0


In [2]:
df_pr=df.copy()
# Điền 0 cho bất kỳ giá trị NaN còn lại nào của previous_year_rating để đảm bảo sạch NaN
df_pr['previous_year_rating'] = df_pr['previous_year_rating'].fillna(0)
df_pr['avg_training_score']=df_pr['avg_training_score'].fillna(df_pr['avg_training_score'].mode()[0])
df_pr.head()

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49.0,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60.0,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50.0,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50.0,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73.0,0


In [3]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Tạo bản sao từ df gốc
df_hot = df_pr.copy()

# 2. Điền giá trị nan cho education trước để tránh lỗi
mode_edu = df_hot['education'].mode()[0]
df_hot['education'] = df_hot['education'].fillna(mode_edu)

# 3. Mã hóa Ordinal cho education (có thứ tự thiết lập sẵn)
ordinal_order = ["Below Secondary", "Bachelor's", "Master's & above"]
edu_encoder = OrdinalEncoder(categories=[ordinal_order])
df_hot['education'] = edu_encoder.fit_transform(df_hot[['education']])

# 4. Mã hóa Ordinal cho region (tự động theo bảng chữ cái để không sinh thêm 33 cột One-Hot)
region_encoder = OrdinalEncoder()
df_hot['region'] = region_encoder.fit_transform(df_hot[['region']])

# 5. Mã hóa One-hot cho các cột phân loại khác (đã loại bỏ 'region')
categorical_value = ['department', 'gender', 'recruitment_channel']
df_hot = pd.get_dummies(df_hot, columns=categorical_value, drop_first=True, dtype=int)

In [4]:
from sklearn.preprocessing import StandardScaler

df_sc = df_hot.copy()
scaler = StandardScaler()
# Chuẩn hóa các cột số
num_cols = ['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'avg_training_score']
df_sc[num_cols] = scaler.fit_transform(df_sc[num_cols])

# Loại bỏ cột employee_id định danh
df_sc.drop(columns='employee_id', inplace=True)
df_sc.head()

,region,education,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted,department_Finance,department_HR,department_Legal,department_Operations,department_Procurement,department_R&D,department_Sales & Marketing,department_Technology,gender_m,recruitment_channel_referred,recruitment_channel_sourcing
0,31.0,2.0,-0.415276,0.025598,1.283878,0.500460,0,-1.041152,0,0,0,0,0,0,0,1,0,0,0,1
1,14.0,1.0,-0.415276,-0.627135,1.283878,-0.437395,0,-0.227276,0,0,0,0,1,0,0,0,0,1,0,0
2,10.0,1.0,-0.415276,-0.104948,-0.052623,0.265996,0,-0.967163,0,0,0,0,0,0,0,1,0,1,0,1
3,15.0,1.0,1.226063,0.547785,-1.389124,0.969387,0,-0.967163,0,0,0,0,0,0,0,1,0,1,0,0
4,18.0,1.0,-0.415276,1.331064,-0.052623,-0.906322,0,0.734578,0,0,0,0,0,0,0,0,1,1,0,0


In [5]:
df_sc.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 20 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   region                        54808 non-null  float64
 1   education                     54808 non-null  float64
 2   no_of_trainings               54808 non-null  float64
 3   age                           54808 non-null  float64
 4   previous_year_rating          54808 non-null  float64
 5   length_of_service             54808 non-null  float64
 6   awards_won                    54808 non-null  int64  
 7   avg_training_score            54808 non-null  float64
 8   is_promoted                   54808 non-null  int64  
 9   department_Finance            54808 non-null  int64  
 10  department_HR                 54808 non-null  int64  
 11  department_Legal              54808 non-null  int64  
 12  department_Operations         54808 non-null  int64  
 13  d

### **BƯỚC 3: CÂN BẰNG DỮ LIỆU (DATA BALANCING)**

Vì tập dữ liệu có sự chênh lệch lớn (chỉ ~8.5% số nhân viên được thăng chức), quy trình áp dụng hàng loạt các phương pháp lấy mẫu lại (**Resampling**) để loại bỏ thiên kiến và cải thiện khả năng dự đoán lớp thiểu số:

*   **SMOTE**: Tạo thêm mẫu ảo cho lớp được thăng chức.
*   **SMOTE + Tomek Links**: Kết hợp tạo mẫu mới và loại bỏ các cặp mẫu nhiễu gần nhau.
*   **SMOTE + ENN (Lựa chọn chính)**: Kết hợp tạo mẫu và làm sạch biên bằng quy tắc lân cận, giúp mô hình phân biệt rõ ràng hơn giữa các nhóm.
*   **Random Undersampling**: Giảm bớt số lượng mẫu của nhóm không thăng chức.

Dữ liệu sau đó được chia theo tỷ lệ **70% huấn luyện (Train set)** và **30% kiểm thử (Test set)** để đảm bảo tính khách quan.

In [6]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split

# Định nghĩa các phương pháp lấy mẫu lại (Resampling methods)
resamplers = {
    'SMOTE': SMOTE(random_state=42),
    'SMOTE+Tomek': SMOTETomek(random_state=42),
    'SMOTE+ENN': SMOTEENN(random_state=42),
    'Random Undersampling': RandomUnderSampler(random_state=42)
}
# Tách biến độc lập và biến mục tiêu
X = df_sc.drop('is_promoted', axis=1)
y = df_sc['is_promoted']
# Chạy thử và kiểm tra kích thước dữ liệu sau khi resample
for name, resampler in resamplers.items():
    X_resampled, y_resampled = resampler.fit_resample(X, y)
    print(f"{name:<20}: {X_resampled.shape[0]} mẫu ({y_resampled.value_counts().to_dict()})")

# Theo thiết kế, chúng ta sẽ chọn SMOTE+ENN làm dữ liệu huấn luyện chính cho pipeline tiếp theo
X_bal, y_bal = resamplers['SMOTE+ENN'].fit_resample(X, y)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_bal, y_bal, test_size=0.3, random_state=42, stratify=y_bal
)

print(f"\nĐã chuẩn bị tập dữ liệu huấn luyện chính (SMOTE+ENN): {X_train_f.shape}")

SMOTE               : 100280 mẫu ({0: 50140, 1: 50140})
SMOTE+Tomek         : 99594 mẫu ({0: 49797, 1: 49797})
SMOTE+ENN           : 85154 mẫu ({1: 46587, 0: 38567})
Random Undersampling: 9336 mẫu ({0: 4668, 1: 4668})

Đã chuẩn bị tập dữ liệu huấn luyện chính (SMOTE+ENN): (59607, 19)


### **BƯỚC 4: HUẤN LUYỆN MÔ HÌNH LAI & SO SÁNH (HYBRID FRAMEWORK)**
Chúng ta sẽ cài đặt thư viện TPOT, sau đó xây dựng khung mô hình Lai (Proposed Hybrid Framework) kết hợp **AutoML + KNN + Extra Trees** thông qua cơ chế **Soft Voting**.

In [7]:
!pip install tpot

In [8]:
from tpot import TPOTClassifier
import tpot
import inspect
print(tpot.__version__)
print(inspect.signature(TPOTClassifier))

1.1.0
(search_space='linear', scorers=['roc_auc_ovr'], scorers_weights=[1], cv=10, other_objective_functions=[], other_objective_functions_weights=[], objective_function_names=None, bigger_is_better=True, categorical_features=None, memory=None, preprocessing=False, max_time_mins=60, max_eval_time_mins=10, n_jobs=1, validation_strategy='none', validation_fraction=0.2, early_stop=None, warm_start=False, periodic_checkpoint_folder=None, verbose=2, memory_limit=None, client=None, random_state=None, allow_inner_classifiers=None, **tpotestimator_kwargs)


In [13]:
import tpot.tpot_estimator.estimator as tpot_estimator

# Vô hiệu hóa hoàn toàn việc phát hiện Dask
tpot_estimator._is_dask_enabled = lambda: False
print("Đã vô hiệu hóa Dask trong TPOT.")

Đã vô hiệu hóa Dask trong TPOT.


In [14]:
from tpot import TPOTClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
import numpy as np
import os

print("Đang chạy AutoML TPOT...")
tpot_model = TPOTClassifier(
    scorers=['f1'],
    cv=5,
    max_time_mins=45,
    max_eval_time_mins=5,
    early_stop=10,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

tpot_model.fit(X_train_f, y_train_f)
best_tpot_pipeline = tpot_model.fitted_pipeline_


# 2. Tinh chỉnh các mô hình truyền thống
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_configs = [
    ('Logistic Regression', LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10]}),
    ('Decision Tree', DecisionTreeClassifier(), {'max_depth': list(range(3, 16)), 'min_samples_split': [2, 5, 10]}),
    ('KNN', KNeighborsClassifier(), {'n_neighbors': list(range(3, 16))}),
    ('Extra Trees', ExtraTreesClassifier(random_state=42), {'n_estimators': list(range(50, 301, 50)), 'max_depth': list(range(3, 16))}),
    ('Random Forest', RandomForestClassifier(random_state=42), {'n_estimators': list(range(50, 301, 50)), 'max_depth': list(range(3, 16))}),
    ('XGBoost', XGBClassifier(eval_metric='logloss'), {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]}),
    ('LightGBM', LGBMClassifier(verbose=-1), {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.2]}),
    ('AdaBoost', AdaBoostClassifier(), {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5]}),
    ('ANN (MLP)', MLPClassifier(max_iter=500), {'hidden_layer_sizes': [(50,), (100,), (50, 50)], 'alpha': [0.0001, 0.05]}),
    ('SVM', SVC(probability=True), {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']})
]

models_comparison = {'TPOT AutoML': best_tpot_pipeline}

for name, model, params in model_configs:
    print(f"Đang tinh chỉnh: {name}...")
    search = RandomizedSearchCV(model, params, n_iter=5, cv=cv_strat, scoring='f1', random_state=42, n_jobs=-1)
    search.fit(X_train_f, y_train_f)
    models_comparison[f"{name} (Tuned)"] = search.best_estimator_

# 3. Xây dựng Hybrid Framework
hybrid_proposed = VotingClassifier(
    estimators=[
        ('tpot', best_tpot_pipeline),
        ('knn', models_comparison['KNN (Tuned)']),
        ('et', models_comparison['Extra Trees (Tuned)'])
    ],
    voting='soft',
    weights=[1, 1, 1]
)

models_comparison['Hybrid Proposed (Final)'] = hybrid_proposed
print("\nHoàn tất cấu hình và tinh chỉnh toàn bộ hệ thống mô hình.")

Đang chạy AutoML TPOT...


/usr/local/lib/python3.12/dist-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41221 instead
  warnings.warn(


TimeoutError: No valid workers found

### **BƯỚC 5: ĐÁNH GIÁ VÀ DIỄN GIẢI KẾT QUẢ**
Tính toán các chỉ số Accuracy, Precision, Recall và F1-Score để so sánh hiệu năng giữa mô hình Lai và các mô hình truyền thống.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import pandas as pd

all_method_results = []

# Duyệt qua từng phương pháp Resampling đã định nghĩa ở Bước 3
for res_name, resampler in resamplers.items():
    print(f"--- Đang đánh giá với phương pháp: {res_name} ---")

    # 1. Resample dữ liệu
    X_curr, y_curr = resampler.fit_resample(X, y)
    X_tr, X_te, y_tr, y_te = train_test_split(X_curr, y_curr, test_size=0.3, random_state=42, stratify=y_curr)

    # 2. Đánh giá từng mô hình trên dữ liệu đã resample
    for model_name, model in models_comparison.items():
        # Huấn luyện lại mô hình với tập dữ liệu hiện tại
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        all_method_results.append({
            'Resampling Method': res_name,
            'Model': model_name,
            'F1-Score': f1_score(y_te, y_pred),
            'Accuracy': accuracy_score(y_te, y_pred),
            'Precision': precision_score(y_te, y_pred),
            'Recall': recall_score(y_te, y_pred)
        })

# Tổng hợp kết quả
full_comparison_df = pd.DataFrame(all_method_results).sort_values(by='F1-Score', ascending=False)
display(full_comparison_df.head(10))

# Trực quan hóa so sánh các phương pháp
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))
sns.barplot(data=full_comparison_df, x='Resampling Method', y='F1-Score', hue='Model')
plt.title('SO SÁNH ĐIỂM F1 GIỮA CÁC PHƯƠNG PHÁP RESAMPLING VÀ MODEL')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()